In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the data
sales = pd.read_csv('../data/sales_train.csv')
items = pd.read_csv('../data/items.csv')
shops = pd.read_csv('../data/shops.csv')
test = pd.read_csv('../data/test.csv')

# 2. Clean Extreme Outliers
sales = sales[sales['item_price'] < 100000]
sales = sales[sales['item_cnt_day'] < 1000]
sales = sales[sales['item_price'] > 0]

# 3. Aggregate to Monthly Sales
# The competition target is monthly sales, represented by 'date_block_num'
monthly_sales = sales.groupby(['date_block_num', 'shop_id', 'item_id']).agg({
    'item_cnt_day': 'sum',
    'item_price': 'mean' # Tracking the average price per month can be a useful feature
}).reset_index()

# Rename the column to reflect the monthly aggregation
monthly_sales.rename(columns={'item_cnt_day': 'item_cnt_month'}, inplace=True)

# 4. Target Clipping (Crucial for this competition)
# The evaluation metric (RMSE) is calculated on target values clipped between 0 and 20.
monthly_sales['item_cnt_month'] = monthly_sales['item_cnt_month'].clip(0, 20)

print(monthly_sales.head())

   date_block_num  shop_id  item_id  item_cnt_month  item_price
0               0        0       32             6.0       221.0
1               0        0       33             3.0       347.0
2               0        0       35             1.0       247.0
3               0        0       43             1.0       221.0
4               0        0       51             2.0       128.5


However, there is a hidden trap in this dataset: this groupby only creates rows for items that actually sold in a given month. The test set requires you to predict sales for all active item-shop combinations, even if they sold zero units.

To handle this, you need to generate a "grid" of all possible shop and item combinations for every month and merge your monthly_sales onto it.

   date_block_num  shop_id  item_id  item_cnt_month  item_price
0               0        0       19             0.0         0.0
1               0        0       27             0.0         0.0
2               0        0       28             0.0         0.0
3               0        0       29             0.0         0.0
4               0        0       32             6.0       221.0
Grid size: 10913804 rows


c:\Users\Pradeep\anaconda3\Lib\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


### Inference: The Necessity of the Cartesian Grid Transformation

#### 1. The Machine Learning Implication
* **Preventing Survivorship Bias:** Training a model exclusively on items that successfully sold teaches the algorithm that every item on a shelf will always sell. 
* **Learning Negative Patterns:** Tree-based models like XGBoost and LightGBM require explicit negative examples (`0.0` target values) to learn the conditions that correlate with zero demand (e.g., outdated products, incompatible demographics).
* **Accurate Lag Features:** Injecting these zeros provides the continuous historical timeline required to calculate accurate moving averages. A product dropping from 6 sales to 0 sales is a massive trend indicator that is lost if the row is simply omitted.

#### 2. The Business Implication
* **Inventory Reality:** The dataset exploding from ~1.6 million successful monthly transactions to nearly 11 million rows mathematically demonstrates that the vast majority of active inventory in the catalog sits unsold at most store locations in any given month.
* **The Cold Start Baseline:** This structural sparsity mirrors the exact challenge presented in the test set, which requires predictions on thousands of items that have zero historical sales at specific shops.